# LeetCode 389 — Find the Difference bằng Python

**Link đề:** https://leetcode.com/problems/find-the-difference/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Bit Manipulation, Sorting

```python
class Solution:
    def findTheDifference(self, s: str, t: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu bài: `t` là `s` **xáo trộn** rồi **chèn thêm 1 chữ** — tìm chữ được thêm.
2. Đi qua 5 cách từ dễ đến "mẹo": xóa từng chữ → **sắp xếp** → **đếm tần suất** → **tổng mã ký tự** → **XOR**.
3. Học `ord`/`chr`, phép **XOR** `^` và tính chất `a ^ a = 0` — kỹ thuật "triệt tiêu cặp" rất hay gặp.
4. Trace từng cách; chứng minh tính đúng; so sánh Time/Space.
5. Nhận dạng pattern "**mọi thứ xuất hiện theo cặp, trừ một**".

## 1. Đề bài

Cho hai chuỗi `s` và `t`. Chuỗi `t` được tạo bằng cách **xáo trộn ngẫu nhiên** `s` rồi **chèn thêm một chữ cái** ở vị trí ngẫu nhiên. Trả về chữ cái được thêm.

| `s` | `t` | Output | Vì sao |
|---|---|:---:|---|
| `"abcd"` | `"abcde"` | `"e"` | `e` là chữ thêm vào |
| `""` | `"y"` | `"y"` | `s` rỗng, `t` chỉ có chữ thêm |
| `"a"` | `"aa"` | `"a"` | chữ thêm **có thể trùng** chữ đã có! |

**Constraints:** `0 <= len(s) <= 1000`; `len(t) == len(s) + 1`; chỉ gồm chữ cái thường.

**Bẫy quan trọng:** chữ được thêm có thể là chữ **đã có** trong `s` (ví dụ 3). Vì vậy "tìm chữ của `t` không có trong `s`" là **sai**; phải so **số lần**.

In [1]:
s, t = "a", "aa"
print("Chữ của t không có trong s:", [ch for ch in t if ch not in s], "<- rỗng! cách nghĩ 'không có mặt' thất bại")
print("Số lần 'a': trong s =", s.count("a"), "| trong t =", t.count("a"), "-> 'a' nhiều hơn 1 lần")

Chữ của t không có trong s: [] <- rỗng! cách nghĩ 'không có mặt' thất bại
Số lần 'a': trong s = 1 | trong t = 2 -> 'a' nhiều hơn 1 lần


## 2. Nền tảng Python

### 2.1. `ord` và `chr`

- `ord(ch)` → mã số nguyên của ký tự: `ord('a') = 97`, `ord('e') = 101`.
- `chr(n)` → ký tự có mã `n`: `chr(101) = 'e'`.

Hai hàm là **nghịch đảo** của nhau: `chr(ord(ch)) == ch`.

In [2]:
for ch in "abez":
    print(ch, "-> ord", ord(ch), "-> chr", chr(ord(ch)))

a -> ord 97 -> chr a
b -> ord 98 -> chr b
e -> ord 101 -> chr e
z -> ord 122 -> chr z


### 2.2. `sorted(chuỗi)` trả về **list ký tự** đã sắp xếp

`sorted("dcba")` → `['a', 'b', 'c', 'd']`. Muốn lại chuỗi thì `"".join(...)`.

In [3]:
print(sorted("bdca"))
print("".join(sorted("bdca")))

['a', 'b', 'c', 'd']
abcd


### 2.3. Phép XOR `^` trên số nguyên

XOR so sánh **từng bit**: bit kết quả là `1` nếu hai bit **khác** nhau, `0` nếu **giống** nhau.

| a | b | a ^ b |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Ba tính chất vàng:
1. `x ^ x = 0` (một số XOR chính nó thì triệt tiêu).
2. `x ^ 0 = x`.
3. Giao hoán và kết hợp: thứ tự XOR **không quan trọng**.

⇒ XOR một loạt số mà mọi số xuất hiện **chẵn lần** trừ một số → kết quả là **số lẻ loi** đó.

In [4]:
print("5 ^ 5 =", 5 ^ 5, "| 5 ^ 0 =", 5 ^ 0)
print("bin(5) =", bin(5), "| bin(3) =", bin(3), "| 5 ^ 3 =", 5 ^ 3, "=", bin(5 ^ 3))
print("3 ^ 7 ^ 3 =", 3 ^ 7 ^ 3, "| 7 ^ 3 ^ 3 =", 7 ^ 3 ^ 3, "(thứ tự không quan trọng)")

5 ^ 5 = 0 | 5 ^ 0 = 5
bin(5) = 0b101 | bin(3) = 0b11 | 5 ^ 3 = 6 = 0b110
3 ^ 7 ^ 3 = 7 | 7 ^ 3 ^ 3 = 7 (thứ tự không quan trọng)


`bin(n)` in dạng nhị phân với tiền tố `0b`: `bin(5) = '0b101'`.

## 3. Cách 1 — Trực giác: xóa lần lượt chữ của `s` khỏi `t`

"Cầm" một bản sao của `t` dưới dạng list; với mỗi chữ của `s`, xóa **một** bản của chữ đó khỏi list. Cuối cùng list còn đúng 1 chữ — chính là chữ thêm.

`list.remove(x)` xóa **lần xuất hiện đầu tiên** của `x` (quét list → `O(n)`).

In [5]:
def find_diff_remove(s: str, t: str) -> str:
    remaining = list(t)
    for ch in s:
        remaining.remove(ch)        # O(len) mỗi lần
    return remaining[0]

for s, t in [("abcd", "abcde"), ("", "y"), ("a", "aa"), ("ab", "bba")]:
    print(repr(s), repr(t), "->", find_diff_remove(s, t))

'abcd' 'abcde' -> e
'' 'y' -> y
'a' 'aa' -> a
'ab' 'bba' -> b


### 3.1. Nút thắt

Mỗi `remove` quét list → tổng `O(n²)`. Với `n = 1000` vẫn qua, nhưng ta muốn hiểu cách tốt hơn. Câu hỏi thực sự là: **chữ nào xuất hiện trong `t` nhiều hơn trong `s` một lần?** → so sánh **multiset** (đếm), hoặc dùng mẹo **toán học** để "trừ" hết các chữ chung.

## 4. Cách 2 — Sắp xếp rồi so từng vị trí

Sắp xếp cả hai; vị trí **đầu tiên** khác nhau chứa chữ thêm. Nếu không khác ở `len(s)` vị trí đầu, chữ thêm là **chữ cuối** của `t` đã sắp.

In [6]:
def find_diff_sort(s: str, t: str) -> str:
    a, c = sorted(s), sorted(t)
    for i in range(len(a)):
        if a[i] != c[i]:
            return c[i]
    return c[-1]

for s, t in [("abcd", "abcde"), ("", "y"), ("a", "aa"), ("ab", "bba"), ("xyz", "xayz")]:
    print(repr(s), repr(t), "->", find_diff_sort(s, t))

'abcd' 'abcde' -> e
'' 'y' -> y
'a' 'aa' -> a
'ab' 'bba' -> b
'xyz' 'xayz' -> a


In [7]:
def trace_sort(s, t):
    a, c = sorted(s), sorted(t)
    print("sorted(s) =", a, "| sorted(t) =", c)
    for i in range(len(a)):
        print(f"  i={i}: {a[i]} vs {c[i]}", "-> KHÁC, trả " + c[i] if a[i] != c[i] else "-> giống")
        if a[i] != c[i]:
            return c[i]
    print("  Không khác trong", len(a), "vị trí đầu -> trả chữ cuối", c[-1])
    return c[-1]

_ = trace_sort("bca", "acbb")
_ = trace_sort("abc", "cabd")

sorted(s) = ['a', 'b', 'c'] | sorted(t) = ['a', 'b', 'b', 'c']
  i=0: a vs a -> giống
  i=1: b vs b -> giống
  i=2: c vs b -> KHÁC, trả b
sorted(s) = ['a', 'b', 'c'] | sorted(t) = ['a', 'b', 'c', 'd']
  i=0: a vs a -> giống
  i=1: b vs b -> giống
  i=2: c vs c -> giống
  Không khác trong 3 vị trí đầu -> trả chữ cuối d


**Vì sao vị trí khác đầu tiên đúng là chữ thêm?** Trước vị trí đó hai list giống hệt. Chữ thêm `x` được "chen" vào list sắp xếp của `s` tại vị trí của nó, đẩy mọi chữ phía sau sang phải 1 ô; nên tại chỗ chen, `c[i] = x` còn `a[i]` là chữ lớn hơn `x` (hoặc không tồn tại → chữ cuối).

## 5. Cách 3 — Đếm tần suất

Đếm `t` rồi trừ đi số đếm của `s`; chữ còn số đếm **1** là chữ thêm. `Counter(t) - Counter(s)` cho đúng một khóa.

In [8]:
from collections import Counter

def find_diff_counter(s: str, t: str) -> str:
    diff = Counter(t) - Counter(s)
    return next(iter(diff))          # lấy khóa duy nhất

def find_diff_dict(s: str, t: str) -> str:
    cnt = {}
    for ch in t:
        cnt[ch] = cnt.get(ch, 0) + 1
    for ch in s:
        cnt[ch] -= 1                 # ch chắc chắn có trong cnt vì t chứa mọi chữ của s
    for ch, k in cnt.items():
        if k == 1:
            return ch

print(Counter("aab") - Counter("ab"))
for s, t in [("abcd", "abcde"), ("", "y"), ("a", "aa")]:
    print(repr(s), repr(t), "->", find_diff_counter(s, t), find_diff_dict(s, t))

Counter({'a': 1})
'abcd' 'abcde' -> e e
'' 'y' -> y y
'a' 'aa' -> a a


`iter(x)` tạo **bộ duyệt**; `next(...)` lấy phần tử **đầu tiên** của nó. Với dict/Counter, đó là khóa đầu tiên.

## 6. Cách 4 — Tổng mã ký tự (sum of `ord`)

Mỗi chữ của `s` cũng có trong `t`. Nếu cộng mã `ord` của mọi chữ trong `t` rồi trừ tổng của `s`, các chữ chung **triệt tiêu**, còn lại đúng mã của chữ thêm.

```text
s = "abcd": 97 + 98 + 99 + 100 = 394
t = "abcde": 394 + 101 = 495
495 - 394 = 101 -> chr(101) = 'e'
```

In [9]:
def find_diff_sum(s: str, t: str) -> str:
    return chr(sum(ord(c) for c in t) - sum(ord(c) for c in s))

print(sum(ord(c) for c in "abcd"), sum(ord(c) for c in "abcde"))
for s, t in [("abcd", "abcde"), ("", "y"), ("a", "aa")]:
    print(repr(s), repr(t), "->", find_diff_sum(s, t))

394 495
'abcd' 'abcde' -> e
'' 'y' -> y
'a' 'aa' -> a


Trong Python số nguyên không tràn. Ở C++/Java, tổng có thể tràn nếu chuỗi rất dài — khi đó dùng XOR (Cách 5) hoặc trừ dần trong vòng lặp.

## 7. Cách 5 — XOR mọi ký tự (lời giải "đẹp" nhất)

XOR mã của **mọi** ký tự trong `s` và `t`. Mỗi chữ chung xuất hiện **hai lần** (một ở `s`, một ở `t`) → triệt tiêu (`x ^ x = 0`). Còn lại đúng mã chữ thêm.

Kể cả khi chữ thêm trùng chữ đã có (ví dụ `"a"`, `"aa"`): `a` xuất hiện 3 lần → `a ^ a ^ a = a`.

In [10]:
def find_diff_xor(s: str, t: str) -> str:
    x = 0
    for ch in s:
        x ^= ord(ch)
    for ch in t:
        x ^= ord(ch)
    return chr(x)

for s, t in [("abcd", "abcde"), ("", "y"), ("a", "aa"), ("ab", "bba")]:
    print(repr(s), repr(t), "->", find_diff_xor(s, t))

'abcd' 'abcde' -> e
'' 'y' -> y
'a' 'aa' -> a
'ab' 'bba' -> b


## 8. TRACE — Cách 5 với `s = "ab"`, `t = "bba"`

`ord('a') = 97 = 0b1100001`, `ord('b') = 98 = 0b1100010`.

| Bước | Ký tự | Nguồn | `ord` | `x` trước | `x` sau |
|---:|:-:|:-:|---:|---:|---:|
| 1 | a | s | 97 | 0 | 97 |
| 2 | b | s | 98 | 97 | 3 |
| 3 | b | t | 98 | 3 | 97 |
| 4 | b | t | 98 | 97 | 3 |
| 5 | a | t | 97 | 3 | 98 |

Kết quả `x = 98` → `chr(98) = 'b'`. Giá trị trung gian (`3`, `97`) không có nghĩa riêng — chỉ **cuối cùng** mới "sạch", vì mọi cặp đã triệt tiêu.

In [11]:
def trace_xor(s: str, t: str) -> str:
    x = 0
    step = 1
    for src, string in (("s", s), ("t", t)):
        for ch in string:
            before = x
            x ^= ord(ch)
            print(f"Bước {step}: '{ch}' từ {src} | ord={ord(ch)} ({bin(ord(ch))}) | x: {before} -> {x} ({bin(x)})")
            step += 1
    print("Kết quả:", x, "->", repr(chr(x)))
    return chr(x)

_ = trace_xor("ab", "bba")

Bước 1: 'a' từ s | ord=97 (0b1100001) | x: 0 -> 97 (0b1100001)
Bước 2: 'b' từ s | ord=98 (0b1100010) | x: 97 -> 3 (0b11)
Bước 3: 'b' từ t | ord=98 (0b1100010) | x: 3 -> 97 (0b1100001)
Bước 4: 'b' từ t | ord=98 (0b1100010) | x: 97 -> 3 (0b11)
Bước 5: 'a' từ t | ord=97 (0b1100001) | x: 3 -> 98 (0b1100010)
Kết quả: 98 -> 'b'


`for src, string in (("s", s), ("t", t)):` duyệt một **tuple các cặp** — cách gọn để chạy cùng một đoạn code cho hai chuỗi và biết đang ở chuỗi nào.

### 8.1. XOR ngắn gọn với `functools.reduce`

`reduce(f, dãy, khởi_đầu)` "gộp" dãy bằng hàm `f` hai tham số: `f(f(f(khởi_đầu, x1), x2), x3)...`. `operator.xor(a, b)` chính là `a ^ b`.

In [12]:
from functools import reduce
from operator import xor

print(reduce(xor, [3, 7, 3], 0))

def find_diff_reduce(s: str, t: str) -> str:
    return chr(reduce(xor, map(ord, s + t), 0))

print(find_diff_reduce("abcd", "abcde"), find_diff_reduce("a", "aa"))

7
e a


`map(ord, s + t)` áp dụng `ord` cho từng ký tự của chuỗi ghép `s + t` (không tạo list, sinh dần).

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def findTheDifference(self, s: str, t: str) -> str:
        x = 0
        for ch in s:
            x ^= ord(ch)
        for ch in t:
            x ^= ord(ch)
        return chr(x)


sol = Solution()
print(sol.findTheDifference("abcd", "abcde"), sol.findTheDifference("", "y"), sol.findTheDifference("a", "aa"))

e y a


Các bản nộp thay thế:

```python
# Đếm
from collections import Counter
class Solution:
    def findTheDifference(self, s: str, t: str) -> str:
        return next(iter(Counter(t) - Counter(s)))

# Tổng ord
class Solution:
    def findTheDifference(self, s: str, t: str) -> str:
        return chr(sum(map(ord, t)) - sum(map(ord, s)))
```

## 10. Vì sao đúng?

Gọi `x` là chữ thêm. Khi đó **multiset** chữ của `t` = multiset của `s` + `{x}`.

- **Đếm:** `count_t(c) − count_s(c)` bằng `1` nếu `c = x`, bằng `0` nếu không → khóa duy nhất còn lại là `x`.
- **Tổng ord:** `Σ ord(t) − Σ ord(s) = ord(x)` vì mọi chữ khác góp mặt cùng số lần ở hai bên.
- **XOR:** mỗi chữ `c ≠ x` xuất hiện `2·count_s(c)` lần (chẵn) → XOR triệt tiêu; `x` xuất hiện `2·count_s(x) + 1` lần (lẻ) → còn lại `ord(x)`. Tính giao hoán/kết hợp cho phép "gom" các bản sao lại dù thứ tự bị xáo trộn.
- **Sắp xếp:** đã giải thích ở mục 4.

## 11. Độ phức tạp

Gọi `n = len(s)` (`len(t) = n + 1`).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. `list.remove` | `O(n²)` | `O(n)` | trực giác |
| 2. Sắp xếp | `O(n log n)` | `O(n)` | dễ chứng minh |
| 3. Đếm (dict/Counter) | `O(n)` | `O(Σ)` = `O(26)` | tổng quát, dễ mở rộng |
| 4. Tổng `ord` | `O(n)` | `O(1)` | cẩn thận tràn số ở ngôn ngữ khác |
| 5. XOR | `O(n)` | `O(1)` | không tràn, "đẹp" nhất |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
methods = {
    "remove": find_diff_remove,
    "sort": find_diff_sort,
    "counter": find_diff_counter,
    "dict": find_diff_dict,
    "sum": find_diff_sum,
    "xor": find_diff_xor,
    "reduce": find_diff_reduce,
    "Solution": Solution().findTheDifference,
}
cases = [
    ("abcd", "abcde", "e"), ("", "y", "y"),           # mẫu đề
    ("a", "aa", "a"),                                 # chữ thêm trùng chữ có sẵn
    ("ae", "aea", "a"),
    ("abc", "zabc", "z"),                             # chữ thêm ở đầu
    ("zzz", "zzzz", "z"),
    ("xyz", "xayz", "a"),                             # chữ thêm nhỏ hơn mọi chữ
]
for s, t, exp in cases:
    for name, f in methods.items():
        assert f(s, t) == exp, (name, s, t)
    print(f"{s!r:>7} {t!r:>8} -> {exp!r} PASS")

 'abcd'  'abcde' -> 'e' PASS
     ''      'y' -> 'y' PASS
    'a'     'aa' -> 'a' PASS
   'ae'    'aea' -> 'a' PASS
  'abc'   'zabc' -> 'z' PASS
  'zzz'   'zzzz' -> 'z' PASS
  'xyz'   'xayz' -> 'a' PASS


In [15]:
import random, string
rng = random.Random(389)
for _ in range(1000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(0, 10)))
    extra = rng.choice("abcd")
    chars = list(s)
    chars.insert(rng.randint(0, len(chars)), extra)   # chèn ở vị trí ngẫu nhiên
    rng.shuffle(chars)                                  # xáo trộn
    t = "".join(chars)
    for name, f in methods.items():
        assert f(s, t) == extra, (name, s, t, extra)
print("1000 test ngẫu nhiên (đúng cách sinh dữ liệu của đề) PASS")

1000 test ngẫu nhiên (đúng cách sinh dữ liệu của đề) PASS


`list.insert(i, x)` chèn `x` vào vị trí `i`; `rng.shuffle(list)` xáo trộn **tại chỗ**. Đây là cách sinh test **giống hệt mô tả đề**, nên đáp án `extra` biết trước — một kiểu kiểm thử rất mạnh.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tìm chữ "không có mặt" trong `s`
Sai khi chữ thêm trùng chữ cũ (`"a"`, `"aa"` → không tìm thấy gì).

### Lỗi 2 — So sánh theo vị trí mà **không sắp xếp**
`t` đã bị xáo trộn, so `s[i]` với `t[i]` vô nghĩa.

### Lỗi 3 — Sắp xếp nhưng quên trường hợp chữ thêm nằm **cuối**
Vòng lặp không tìm thấy khác biệt → hàm trả `None`.

### Lỗi 4 — Trả về số thay vì ký tự
Quên `chr(...)` → trả `101` thay vì `'e'`.

In [16]:
def wrong_presence(s, t):
    for ch in t:
        if ch not in s:
            return ch

def wrong_positional(s, t):
    for i in range(len(s)):
        if s[i] != t[i]:
            return t[i]
    return t[-1]

def wrong_sort_no_last(s, t):
    a, c = sorted(s), sorted(t)
    for i in range(len(a)):
        if a[i] != c[i]:
            return c[i]

def wrong_no_chr(s, t):
    x = 0
    for ch in s + t:
        x ^= ord(ch)
    return x

print("Lỗi 1:", wrong_presence("a", "aa"), "(đúng: 'a')")
print("Lỗi 2:", repr(wrong_positional("abcd", "dcbae")), "(đúng: 'e')")
print("Lỗi 3:", wrong_sort_no_last("abcd", "abcde"), "(đúng: 'e')")
print("Lỗi 4:", wrong_no_chr("abcd", "abcde"), "(đúng: 'e')")

Lỗi 1: None (đúng: 'a')
Lỗi 2: 'd' (đúng: 'e')
Lỗi 3: None (đúng: 'e')
Lỗi 4: 101 (đúng: 'e')


## 14. Mẹo nhận dạng pattern

- "Mọi phần tử xuất hiện **theo cặp/chẵn lần**, trừ một" → **XOR** toàn bộ.
- "Hai tập gần giống nhau, chênh **một** phần tử" → **hiệu tổng** hoặc XOR, bộ nhớ `O(1)`.
- "Cần biết **phần tử nào** khác về số lần" (có thể nhiều phần tử) → **đếm** (Counter).
- Bài anh em: **136 Single Number** (XOR), **268 Missing Number** (tổng hoặc XOR với chỉ số), **448 Find All Numbers Disappeared**, **242 Valid Anagram**, **1460**.

## 15. Biến thể

### 15.1. Single Number (LeetCode 136)

Mọi số xuất hiện hai lần trừ một số. XOR tất cả.

In [17]:
def single_number(nums: list[int]) -> int:
    x = 0
    for v in nums:
        x ^= v
    return x

print(single_number([4, 1, 2, 1, 2]), single_number([7]))

4 7


### 15.2. Missing Number (LeetCode 268)

Mảng chứa `n` số khác nhau trong `[0, n]`, thiếu đúng một. Tổng kỳ vọng `n(n+1)/2` trừ tổng thực tế — chính là "Find the Difference" phiên bản số.

In [18]:
def missing_number(nums: list[int]) -> int:
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)

print(missing_number([3, 0, 1]), missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]))

2 8


### 15.3. `t` thêm **hai** chữ

XOR không còn đủ (cho `x ^ y`, không tách được ngay); đếm thì vẫn chạy: `Counter(t) - Counter(s)`.

In [19]:
def find_two_added(s: str, t: str) -> list[str]:
    return sorted((Counter(t) - Counter(s)).elements())

print(find_two_added("abc", "cbxad"), find_two_added("a", "aaa"))

['d', 'x'] ['a', 'a']


## 16. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** Trace Cách 4 (tổng `ord`) cho `s = "xyz"`, `t = "zxyw"`. Hiệu bằng bao nhiêu? Chữ nào?

**Bài 2 (dự đoán output).** `6 ^ 3`, `6 ^ 3 ^ 3`, và `reduce(xor, [1, 2, 3, 1, 2], 0)` bằng bao nhiêu?

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_xor`.

**Bài 4 (viết lại).** Viết bản "tổng `ord`" nhưng **không** tính hai tổng lớn riêng: dùng **một** biến, cộng `ord` của `t` và trừ `ord` của `s` trong cùng vòng lặp `for i in range(len(s))`, rồi cộng ký tự cuối của `t`.

**Bài 5 (biến thể).** `find_removed(s, t)`: `t` là `s` xáo trộn rồi **xóa** một chữ. Trả chữ bị xóa.

**Bài 6 (mini challenge).** Mảng số nguyên mà mọi số xuất hiện **ba lần** trừ một số xuất hiện **một lần**. Tìm số đó với bộ nhớ `O(1)` (gợi ý: đếm từng bit modulo 3 — LeetCode 137). Có thể giả sử số không âm.

In [20]:
def exercise_xor(s: str, t: str) -> str:
    # TODO 1: x = 0
    # TODO 2: XOR ord của mọi ký tự trong s và t
    # TODO 3: return chr(x)
    pass

# assert exercise_xor("abcd", "abcde") == "e" and exercise_xor("a", "aa") == "a"

In [21]:
def exercise_one_variable(s: str, t: str) -> str:
    # TODO
    pass

# assert exercise_one_variable("abcd", "abcde") == "e" and exercise_one_variable("", "y") == "y"

In [22]:
def find_removed(s: str, t: str) -> str:
    # TODO
    pass

# assert find_removed("abcde", "dcba") == "e" and find_removed("aa", "a") == "a"

In [23]:
def single_number_ii(nums: list[int]) -> int:
    # TODO: với mỗi bit 0..31, đếm số phần tử có bit đó = 1; lấy modulo 3
    pass

# assert single_number_ii([2, 2, 3, 2]) == 3 and single_number_ii([0, 1, 0, 1, 0, 1, 99]) == 99

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | cộng mã của t, trừ mã của s | x=120, y=121, z=122 | hiệu là `ord('w') = 119` |
| 2 | dùng bảng bit | 6=110, 3=011 | `x ^ x = 0` |
| 3 | xem Cách 5 | `x ^= ord(ch)` | hai vòng hoặc vòng trên `s + t` |
| 4 | một biến `total` | `total += ord(t[i]) - ord(s[i])` | cuối cùng `total += ord(t[-1])` |
| 5 | đổi vai `s` và `t` | chữ bị xóa = chữ thêm khi nhìn ngược | gọi lại `find_diff_xor(t, s)` |
| 6 | XOR không dùng được (3 là lẻ) | đếm bit `i` qua mọi số | bit kết quả = `count % 3` |

## 18. Đáp án tham khảo + Template + Checklist

**Bài 1:** Σt = 122 + 120 + 121 + 119 = 482; Σs = 120 + 121 + 122 = 363; hiệu `119` → `'w'`.

**Bài 2:** `6 ^ 3 = 5` (110 ^ 011 = 101); `6 ^ 3 ^ 3 = 6`; `reduce(...) = 3`.

In [24]:
def answer_xor(s, t):
    x = 0
    for ch in s + t:
        x ^= ord(ch)
    return chr(x)

def answer_one_variable(s, t):
    total = 0
    for i in range(len(s)):
        total += ord(t[i]) - ord(s[i])
    total += ord(t[-1])
    return chr(total)

def answer_find_removed(s, t):
    return find_diff_xor(t, s)

def answer_single_number_ii(nums):
    result = 0
    for bit in range(32):
        count = sum((v >> bit) & 1 for v in nums)     # >> dịch phải, & 1 lấy bit cuối
        if count % 3:
            result |= 1 << bit                         # bật bit thứ 'bit'
    return result

print("Bài 1:", sum(map(ord, "zxyw")), "-", sum(map(ord, "xyz")), "=", sum(map(ord, "zxyw")) - sum(map(ord, "xyz")), "->", repr(find_diff_sum("xyz", "zxyw")))
print("Bài 2:", 6 ^ 3, 6 ^ 3 ^ 3, reduce(xor, [1, 2, 3, 1, 2], 0))
assert answer_xor("abcd", "abcde") == "e" and answer_xor("a", "aa") == "a"
assert answer_one_variable("abcd", "abcde") == "e" and answer_one_variable("", "y") == "y"
assert answer_find_removed("abcde", "dcba") == "e" and answer_find_removed("aa", "a") == "a"
assert answer_single_number_ii([2, 2, 3, 2]) == 3 and answer_single_number_ii([0, 1, 0, 1, 0, 1, 99]) == 99
print("Đáp án PASS toàn bộ!")

Bài 1: 482 - 363 = 119 -> 'w'
Bài 2: 5 6 3
Đáp án PASS toàn bộ!


(Bài 6: `v >> bit` dịch `v` sang phải `bit` vị trí; `& 1` lấy bit thấp nhất; `1 << bit` là số chỉ có bit thứ `bit` bằng 1; `|=` bật bit đó trong `result`.)

### Template nhớ nhanh

```python
class Solution:
    def findTheDifference(self, s: str, t: str) -> str:
        x = 0
        for ch in s + t:          # mọi chữ chung xuất hiện chẵn lần -> triệt tiêu
            x ^= ord(ch)
        return chr(x)             # còn lại đúng chữ được thêm
```

### Checklist submit
- [ ] Xử lý trường hợp chữ thêm **trùng** chữ đã có (`"a"`, `"aa"`)?
- [ ] `s` rỗng vẫn đúng (`""`, `"y"`)?
- [ ] Trả về **ký tự** (`chr`), không phải số?
- [ ] Nếu dùng sắp xếp: có trả `c[-1]` khi chữ thêm nằm cuối?
- [ ] Giải thích được vì sao XOR/tổng triệt tiêu các chữ chung?